In [ ]:
from typing import Tuple, OrderedDict
import torch
import torch.nn as nn 
import torch.nn.functional as F

def forward_hook(self, input, output):
    if type(input[0]) in (list, tuple):
        self.X = []
        for i in input[0]:
            x = i.detach()
            x.requires_grad = True
            self.X.append(x)
    else:
        self.X = input[0].detach()
        self.X.requires_grad = True
    self.Y = output

def backward_hook(self, grad_input, grad_output):
    self.grad_input = grad_input
    self.grad_output = grad_output

def safe_divide(a, b):
    den = b
    den = den + den.eq(0).type(den.type()) * 1e-9
    return a / (den)

class RelProp(nn.Module):
    def __init__(self) -> None:
        super(RelProp, self).__init__()
        self.register_forward_hook(forward_hook)

    # S is upstream gradient flow wrt to Z. Z is a function of X
    # C is gradient flow wrt to X now. 
    def gradprop(self, Z, X, S):
        C = torch.autograd.grad(Z,X,S, retain_graph=True)
        return C

    # Forwards and does nothing
    def relprop(self, R, alpha):
        return R

class RelPropSimple(RelProp):
    # General Deep Taylor Decomposition
    def relprop(self, R, alpha):
        Z = self.forward(self.X)
        S = safe_divide(R, Z)
        C = self.gradprop(Z, self.X, S)
        
        if torch.is_tensor(self.X) == False:
            outputs = []
            outputs.append(self.X[0] * C[0])
            outputs.append(self.X[1] * C[1])
        else:
            outputs = self.X * (C[0])
        return outputs

class AddEye(RelPropSimple):
    def forward(self, input):
        return input + torch.eye(input.shape[2]).expand_as(input).to(input.device)

class ReLU(nn.ReLU, RelProp):
    pass

class GELU(nn.GELU, RelProp):
    pass

class Softmax(nn.Softmax, RelProp):
    pass

class LayerNorm(nn.LayerNorm, RelProp):
    pass

class Dropout(nn.Dropout, RelProp):
    pass

class MaxPool2d(nn.MaxPool2d, RelPropSimple):
    pass

class AdaptiveAvgPool2d(nn.AdaptiveAvgPool2d, RelPropSimple):
    pass


class AvgPool2d(nn.AvgPool2d, RelPropSimple):
    pass


# Understood now. Normalization of Addition operation
class Add(RelPropSimple):
    def forward(self, inputs):
        return torch.add(*inputs)
    
    def relprop(self, R, alpha):
        Z = self.forward(self.X)
        S = safe_divide(R, Z)
        C = self.gradprop(Z, self.X, S)
        a = self.X[0] * C[0]
        b = self.X[1] * C[1]

        a_sum = a.sum()
        b_sum = b.sum()

        a_fact = safe_divide(a_sum.abs(), a_sum.abs() + b_sum.abs()) * R.sum()
        b_fact = safe_divide(b_sum.abs(), a_sum.abs() + b_sum.abs()) * R.sum()

        a = a * safe_divide(a_fact, a.sum())
        b = b * safe_divide(b_fact, b.sum())

        outputs = [a, b]
        return outputs
    
class Cat(RelProp):
    def forward(self, inputs, dim):
        self.dim = dim
        return torch.cat(inputs, dim)
    def relprop(self, R, alpha):
        Z = self.forward(self.X, self.dim)
        S = safe_divide(R, Z)
        C = self.gradprop(Z, self.X, S)
        return [ x * c for x, c in zip(self.X, C)]

class einsum(RelPropSimple):
    def __init__(self, equation):
        super().__init__()
        self.equation = equation
    def forward(self, *operands):
        return torch.einsum(self.equation, *operands)

class IndexSelect(RelProp):
    def forward(self, inputs, dim, indices):
        self.__setattr__('dim', dim)
        self.__setattr__('indices', indices)

        return torch.index_select(inputs, dim, indices)

    def relprop(self, R, alpha):
        Z = self.forward(self.X, self.dim, self.indices)
        S = safe_divide(R, Z)
        C = self.gradprop(Z, self.X, S)

        if torch.is_tensor(self.X) == False:
            outputs = []
            outputs.append(self.X[0] * C[0])
            outputs.append(self.X[1] * C[1])
        else:
            outputs = self.X * (C[0])
        return outputs

class Clone(RelProp):
    def forward(self, input, num):
        self.__setattr__('num', num)
        outputs = []
        for _ in range(num):
            outputs.append(input)

        return outputs
    
    def relprop(self, R, alpha):
        Z = []
        for _ in range(self.num):
            Z.append(self.X)
        S = [safe_divide(r, z) for r, z in zip(R, Z)]
        C = self.gradprop(Z, self.X, S)[0]
        R = self.X * C
        return R
# Cat, Batchnorm

class Sequential(nn.Sequential):
    def relprop(self, R, alpha):
        for m in reversed(self._modules.values()):
            R = m.relprop(R, alpha)
        return R

# BatchNorm2D

class Linear(nn.Linear, RelProp):
    def relprop(self, R, alpha):
        beta = alpha - 1
        pw = torch.clamp(self.weight, min = 0)
        nw = torch.clamp(self.weight, max = 0)
        px = torch.clamp(self.X, min = 0)
        nx = torch.clamp(self.X, max = 0)

        def f(w1, w2, x1, x2):
            Z1 = F.linear(x1, w1)
            Z2 = F.linear(x2, w2)
            S1 = safe_divide(R, Z1 + Z2)
            S2 = safe_divide(R, Z1 + Z2)
            C1 = x1 * torch.autograd.grad(Z1, x1, S1)[0]
            C2 = x2 * torch.autograd.grad(Z2, x2, S2)[0]

            return C1 + C2
        
        activator_relevances = f(pw, nw, px, nx)
        inhibitor_relevances = f(nw, pw, px, nx)

        R = alpha * activator_relevances - beta * inhibitor_relevances
        return R

# Conv2D
class Conv2d(nn.Conv2d, RelProp):
    def gradprop2(self, DY, weight):
        Z = self.forward(self.X)

        output_padding = self.X.size()[2] - (
                (Z.size()[2] - 1) * self.stride[0] - 2 * self.padding[0] + self.kernel_size[0])

        return F.conv_transpose2d(DY, weight, stride=self.stride, padding=self.padding, output_padding=output_padding)

    def relprop(self, R, alpha):
        if self.X.shape[1] == 3:
            pw = torch.clamp(self.weight, min=0)
            nw = torch.clamp(self.weight, max=0)
            X = self.X
            L = self.X * 0 + \
                torch.min(torch.min(torch.min(self.X, dim=1, keepdim=True)[0], dim=2, keepdim=True)[0], dim=3,
                          keepdim=True)[0]
            H = self.X * 0 + \
                torch.max(torch.max(torch.max(self.X, dim=1, keepdim=True)[0], dim=2, keepdim=True)[0], dim=3,
                          keepdim=True)[0]
            Za = torch.conv2d(X, self.weight, bias=None, stride=self.stride, padding=self.padding) - \
                 torch.conv2d(L, pw, bias=None, stride=self.stride, padding=self.padding) - \
                 torch.conv2d(H, nw, bias=None, stride=self.stride, padding=self.padding) + 1e-9

            S = R / Za
            C = X * self.gradprop2(S, self.weight) - L * self.gradprop2(S, pw) - H * self.gradprop2(S, nw)
            R = C
        else:
            beta = alpha - 1
            pw = torch.clamp(self.weight, min=0)
            nw = torch.clamp(self.weight, max=0)
            px = torch.clamp(self.X, min=0)
            nx = torch.clamp(self.X, max=0)

            def f(w1, w2, x1, x2):
                Z1 = F.conv2d(x1, w1, bias=None, stride=self.stride, padding=self.padding)
                Z2 = F.conv2d(x2, w2, bias=None, stride=self.stride, padding=self.padding)
                S1 = safe_divide(R, Z1)
                S2 = safe_divide(R, Z2)
                C1 = x1 * self.gradprop(Z1, x1, S1)[0]
                C2 = x2 * self.gradprop(Z2, x2, S2)[0]
                return C1 + C2
            activator_relevances = f(pw, nw, px, nx)
            inhibitor_relevances = f(nw, pw, px, nx)

            R = alpha * activator_relevances - beta * inhibitor_relevances
        return R

class MultiheadAttention(RelProp):
    def __init__(self, embed_dim, num_heads, dropout=0.):
        super(MultiheadAttention, self).__init__()
        self.embed_dim = embed_dim
        self.kdim = embed_dim
        self.vdim = embed_dim

        self.num_heads = num_heads
        self.dropout = Dropout(dropout)
        self.head_dim = embed_dim // num_heads

        self.q_proj = Linear(embed_dim, embed_dim)
        self.k_proj = Linear(embed_dim, embed_dim)
        self.v_proj = Linear(embed_dim, embed_dim)
        self.out_proj = Linear(embed_dim, embed_dim, bias=True)

        self.softmax = Softmax(dim=-1)

        self.einsum1 = einsum('bid,bjd->bij')
        self.einsum2 = einsum('bij,bjd->bid')

        self._register_load_state_dict_pre_hook(MultiheadAttention._pre_load_state_dict)

        self.attn_cam = None
        self.attn = None
        self.attn_gradients = None

    def save_attn_cam(self, cam):
        self.attn_cam = cam

    def get_attn_cam(self):
        return self.attn_cam

    def save_attn(self, attn):
        self.attn = attn

    def get_attn(self):
        return self.attn

    def save_attn_gradients(self, attn_gradients):
        self.attn_gradients = attn_gradients

    def get_attn_gradients(self):
        return self.attn_gradients

    @staticmethod
    def _pre_load_state_dict(state_dict: OrderedDict, prefix, local_metadata, strict,
                              missing_keys, unexpected_keys, error_msgs):
        w = state_dict[prefix + 'in_proj_weight']
        b = state_dict[prefix + 'in_proj_bias']

        embed_dim = w.shape[1]

        state_dict[prefix + 'q_proj.weight'] = w[:embed_dim]
        state_dict[prefix + 'q_proj.bias'] = b[:embed_dim]

        state_dict[prefix + 'k_proj.weight'] = w[embed_dim:2*embed_dim]
        state_dict[prefix + 'k_proj.bias'] = b[embed_dim:2*embed_dim]

        state_dict[prefix + 'v_proj.weight'] = w[2*embed_dim:]
        state_dict[prefix + 'v_proj.bias'] = b[2*embed_dim:]

    def forward(self, query, key, value, key_padding_mask=None,
                need_weights=True, attn_mask=None):
        tgt_len, bsz, embed_dim = query.size()
        src_len, _, _ = key.size()

        self.tgt_len = tgt_len
        self.src_len = src_len
        self.bsz = bsz

        head_dim = embed_dim // self.num_heads
        scaling = float(head_dim) ** -0.5

        self.head_dim = head_dim

        q = self.q_proj(query)
        k = self.k_proj(key)
        v = self.v_proj(value)

        q = q * scaling

        q = q.contiguous().view(tgt_len, bsz * self.num_heads, head_dim).transpose(0, 1)
        k = k.contiguous().view(-1, bsz * self.num_heads, head_dim).transpose(0, 1)
        v = v.contiguous().view(-1, bsz * self.num_heads, head_dim).transpose(0, 1)  # BHxSxD

        # attn_output_weights = torch.bmm(q, k.transpose(1, 2))
        attn_output_weights = self.einsum1([q, k])  # BHxTxS

        attn_output_weights = self.softmax(attn_output_weights)
        attn_output_weights = self.dropout(attn_output_weights)

        self.save_attn(attn_output_weights)
        attn_output_weights.register_hook(self.save_attn_gradients)

        # attn_output = torch.bmm(attn_output_weights, v)
        attn_output = self.einsum2([attn_output_weights, v])  # BHxTxD

        #  BHxTxD -> TxBHxD -> TxBxHD
        attn_output = attn_output.transpose(0, 1).contiguous().view(tgt_len, bsz, embed_dim)
        attn_output = self.out_proj(attn_output)

        return attn_output

    def relprop(self, cam_attn_output, alpha, **kwargs):
        cam_attn_output = self.out_proj.relprop(cam_attn_output, alpha, **kwargs)
        cam_attn_output = cam_attn_output.view(self.tgt_len, self.bsz*self.num_heads, self.head_dim).transpose(0, 1)
        cam_attn_output_weights, cam_v = self.einsum2.relprop(cam_attn_output, alpha, **kwargs)
        cam_attn_output_weights /= 2
        cam_v /= 2
        self.save_attn_cam(cam_attn_output_weights)
        cam_attn_output_weights = self.dropout.relprop(cam_attn_output_weights, alpha, **kwargs)
        cam_attn_output_weights = self.softmax.relprop(cam_attn_output_weights, alpha, **kwargs)
        cam_q, cam_k = self.einsum1.relprop(cam_attn_output_weights, alpha, **kwargs)
        cam_q /= 2
        cam_k /= 2

        cam_v = cam_v.transpose(0, 1).view(self.src_len, self.bsz, self.num_heads*self.head_dim)
        cam_k = cam_k.transpose(0, 1).view(self.src_len, self.bsz, self.num_heads*self.head_dim)
        cam_q = cam_q.transpose(0, 1).view(self.tgt_len, self.bsz, self.num_heads*self.head_dim)

        pre_cam_v = cam_v.min() == cam_v.max() == 0
        cam_v = self.v_proj.relprop(cam_v, alpha, **kwargs)
        cam_k = self.k_proj.relprop(cam_k, alpha, **kwargs)
        cam_q = self.q_proj.relprop(cam_q, alpha, **kwargs)

        if cam_v.min() == cam_v.max() == 0 and not pre_cam_v:
            cam_k_sum = cam_k.sum()
            cam_q_sum = cam_q.sum()
            cam_k_fact = safe_divide(cam_k_sum.abs(), cam_k_sum.abs() + cam_q_sum.abs()) * cam_attn_output.sum()
            cam_q_fact = safe_divide(cam_q_sum.abs(), cam_k_sum.abs() + cam_q_sum.abs()) * cam_attn_output.sum()

            cam_k = cam_k * safe_divide(cam_k_fact, cam_k.sum())
            cam_q = cam_q * safe_divide(cam_q_fact, cam_q.sum())

        return cam_q, cam_k, cam_v

class TokenEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.vocab_size = vocab_size
        self.linear = Linear(vocab_size, embed_dim, bias = False)
    def forward(self, x):
        one_hot = F.one_hot(x, num_classes=self.vocab_size).float()
        return self.linear(one_hot)
    def relprop(self, R, alpha):
        return self.linear.relprop(R, alpha)

In [ ]:

import torch 
import torch
import torch.nn as nn
import math
import numpy as np
from einops import rearrange
import matplotlib.pyplot as plt 

# Implement load_pretrained, trunc_normal_ (might be part of pytorch now) and to2tuple




def _no_grad_trunc_normal_(tensor, mean, std, a, b):
    def norm_cdf(x):
        return (1. + math.erf(x / math.sqrt(2.))) / 2.

    with torch.no_grad():
        l = norm_cdf((a - mean) / std)
        u = norm_cdf((b - mean) / std)
        tensor.uniform_(2 * l - 1, 2 * u -1)
        tensor.erfinv_()
        tensor.mul_(std * math.sqrt(2.))
        tensor.add_(mean)
        tensor.clamp_(min=a, max = b)
        return tensor
def trunc_normal_(tensor, mean=0., std=1., a=-2., b=2.):
    return _no_grad_trunc_normal_(tensor, mean, std, a, b)

def _cfg(url='', **kwargs):
    return {
        'url': url,
        'num_classes': 1000, 'input_size': (3, 224, 224), 'pool_size': None,
        'crop_pct': .9, 'interpolation': 'bicubic',
        'first_conv': 'patch_embed.proj', 'classifier': 'head',
        **kwargs
    }

default_cfgs = {
    # patch models
    'vit_small_patch16_224': _cfg(
        url='https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/vit_small_p16_224-15ec54c9.pth',
    ),
    'vit_base_patch16_224': _cfg(
        url='https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth',
        mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5),
    ),
    'vit_large_patch16_224': _cfg(
        url='https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_large_p16_224-4ee7a4dc.pth',
        mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
}

def compute_rollout_attention(all_layer_matrices, start_layer=0):
    num_tokens = all_layer_matrices[0].shape[1]
    batch_size = all_layer_matrices[0].shape[0]
    eye = torch.eye(num_tokens).expand(batch_size, num_tokens, num_tokens).to(all_layer_matrices[0].device)
    all_layer_matrices = [all_layer_matrices[i] + eye for i in range(len(all_layer_matrices))]
    joint_attention = all_layer_matrices[start_layer]
    for i in range(start_layer + 1, len(all_layer_matrices)):
        joint_attention = all_layer_matrices[i].bmm(joint_attention)
    return joint_attention


class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.) -> None:
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = Linear(in_features, hidden_features)
        self.act = GELU()
        self.fc2 = Linear(hidden_features, out_features)
        self.drop = Dropout(drop)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x
    
    def relprop(self, cam, **kwargs):
        cam = self.drop.relprop(cam, **kwargs)
        cam = self.fc2.relprop(cam, **kwargs)
        cam = self.act.relprop(cam, **kwargs)
        cam = self.fc1.relprop(cam, **kwargs)
        return cam

class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0., proj_drop=0.) -> None:
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        # A = Q*K^T
        self.matmul1 = einsum('bhid,bhjd->bhij')
        # attn = A*V
        self.matmul2 = einsum('bhij,bhjd->bhid')

        self.qkv = Linear(dim, dim * 3, bias = qkv_bias)
        self.attn_drop = Dropout(attn_drop)
        self.proj = Linear(dim, dim)
        self.proj_drop = Dropout(proj_drop)
        self.softmax = Softmax(dim=-1)

        self.attn_cam = None
        self.attn = None
        self.v = None
        self.v_cam = None
        self.attn_gradients = None
    
    def get_attn(self):
        return self.attn
    
    def save_attn(self, attn):
        self.attn = attn
    def save_attn_cam(self, cam):
        self.attn_cam = cam
    def get_attn_cam(self):
        return self.attn_cam
    def get_v(self):
        return self.v
    def save_v(self, v):
        self.v = v
    def save_v_cam(self, cam):
        self.v_cam = cam

    def get_v_cam(self):
        return self.v_cam

    def save_attn_gradients(self, attn_gradients):
        self.attn_gradients = attn_gradients

    def get_attn_gradients(self):
        return self.attn_gradients

    def forward(self, x):
        b, n, _, h = *x.shape, self.num_heads
        qkv = self.qkv(x)
        q, k, v = rearrange(qkv, 'b n (qkv h d) -> qkv b h n d', qkv=3, h=h)
        self.save_v(v)
        dots = self.matmul1([q, k]) * self.scale
        attn = self.softmax(dots)
        attn = self.attn_drop(attn)
        self.save_attn(attn)
        attn.register_hook(self.save_attn_gradients)
        out = self.matmul2([attn, v])
        out = rearrange(out, 'b h n d -> b n (h d)')
        out = self.proj(out)
        out = self.proj_drop(out)
        return out

    def relprop(self, cam, **kwargs):
        cam = self.proj_drop.relprop(cam, **kwargs)
        cam = self.proj.relprop(cam, **kwargs)
        cam = rearrange(cam, 'b n (h d) -> b h n d', h=self.num_heads)

        # attn = A*V
        (cam1, cam_v) = self.matmul2.relprop(cam, **kwargs)
        cam1 /= 2
        cam_v /= 2

        self.save_v_cam(cam_v)
        self.save_attn_cam(cam1)
        cam1 = self.attn_drop.relprop(cam1, **kwargs)
        cam1 = self.softmax.relprop(cam1, **kwargs)

        # A = Q *K^T
        (cam_q, cam_k) = self.matmul1.relprop(cam1, **kwargs)
        cam_q /= 2
        cam_k /= 2

        cam_qkv = rearrange([cam_q, cam_k, cam_v], 'qkv b h n d -> b n (qkv h d)', qkv=3, h=self.num_heads)
        return self.qkv.relprop(cam_qkv, **kwargs)


class Block(nn.Module):
    
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, drop=0., attn_drop=0.):
        super().__init__()
        self.norm1 = LayerNorm(dim, eps=1e-6)
        self.attn = Attention(
            dim, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop
        )
        self.norm2 = LayerNorm(dim, eps=1e-6)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, drop=drop)
        self.add1 = Add()
        self.add2 = Add()
        self.clone1 = Clone()
        self.clone2 = Clone()

    def forward(self, x):
        x1, x2 = self.clone1(x, 2)
        x = self.add1([x1, self.attn(self.norm1(x2))])
        x1, x2 = self.clone2(x, 2)
        x = self.add2([x1, self.mlp(self.norm2(x2))])
        return x

    def relprop(self, cam, **kwargs):
        (cam1, cam2) = self.add2.relprop(cam, **kwargs)
        cam2 = self.mlp.relprop(cam2, **kwargs)
        cam2 = self.norm2.relprop(cam2, **kwargs)
        cam = self.clone2.relprop((cam1, cam2), **kwargs)

        (cam1, cam2) = self.add1.relprop(cam, **kwargs)
        cam2 = self.attn.relprop(cam2, **kwargs)
        cam2 = self.norm1.relprop(cam2, **kwargs)
        cam = self.clone1.relprop((cam1, cam2), **kwargs)

        return cam


class PatchEmbed(nn.Module):
    
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim = 768):
        super().__init__()
        img_size  = (img_size, img_size)
        patch_size = (patch_size, patch_size)
        num_patches = (img_size[1] // patch_size[1]) * (img_size[0] // patch_size[0])
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = num_patches
        self.proj = Conv2d(in_chans, embed_dim, kernel_size = patch_size, stride=patch_size)

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.proj(x).flatten(2).transpose(1, 2)
        return x

    def relprop(self, cam, **kwargs):
        cam = cam.transpose(1,2)
        cam = cam.reshape(cam.shape[0], cam.shape[1],
            (self.img_size[0] // self.patch_size[0]), (self.img_size[1] // self.patch_size[1]))
        return self.proj.relprop(cam, **kwargs)

class VitTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=1000, embed_dim=768, depth=12,
                num_heads=12, mlp_ratio=4., qkv_bias=False,mlp_head=False, drop_rate=0., attn_drop_rate=0.):
        super().__init__()
        self.num_classes = num_classes
        self.num_features = self.embed_dim = embed_dim
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim
        )
        num_patches = self.patch_embed.num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        self.blocks = nn.ModuleList([
            Block(
                dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias,
                drop=drop_rate, attn_drop=attn_drop_rate)
            for i in range(depth)])

        self.norm = LayerNorm(embed_dim)
        if mlp_head:
            self.head = Mlp(embed_dim, int(embed_dim * mlp_ratio), num_classes)
        else:
            self.head = Linear(embed_dim, num_classes)

        trunc_normal_(self.pos_embed, std=.02)
        trunc_normal_(self.cls_token, std=.02)
        self.apply(self._init_weights)

        self.pool = IndexSelect()
        self.add = Add()
        self.inp_grad = None
        
    def save_inp_grag(self, grad):
        self.inp_grad = grad
    def get_inp_grad(self, grad):
        return self.inp_grad

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        
    @property
    def no_weight_decay(self):
        return {'pos_embed', 'cls_token'}
    def forward_features(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim = 1)
        x.register_hook(self.save_inp_grag)

        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x
    def forward(self, x):
        x = self.forward_features(x)
        x = self.pool(x, dim=1, indices=torch.tensor(0, device=x.device))
        x = x.squeeze(1)
        x = self.head(x)
        return x

    def relprop(self, cam=None, method="transformer_attribution", is_ablation=False, start_layer=0, **kwargs):
        cam = self.head.relprop(cam, **kwargs)
        # print("conservation 1", cam.sum())
        cam = self.pool.relprop(cam, **kwargs)
        cam = self.norm.relprop(cam, **kwargs)
        for blk in reversed(self.blocks):
            cam = blk.relprop(cam, **kwargs)
        # print("conservation 2", cam.sum())
        
        if method == "transformer_attribution":
            cams = []
            for blk in self.blocks:
                grad = blk.attn.get_attn_gradients()
                cam = blk.attn.get_attn_cam()
                cam = cam[0].reshape(-1, cam.shape[-1], cam.shape[-1])
                grad = grad[0].reshape(-1, grad.shape[-1], grad.shape[-1])
                cam = grad * cam 
                cam = cam.clamp(min=0).mean(dim=0)
                cams.append(cam.unsqueeze(0))
            rollout = compute_rollout_attention(cams, start_layer=start_layer)
            cam = rollout[:, 0, 1:]
            return cam

        elif method == "rollout":
            attn_cams = []
            for blk in self.blocks:
                attn_heads = blk.attn.get_attn_cam().clamp(min=0)
                avg_heads = (attn_heads.sum(dim=1) / attn_heads.shape[1]).detach()
                attn_cams.append(avg_heads)
            cam = compute_rollout_attention(attn_cams, start_layer=start_layer)
            cam = cam[:, 0, 1: ]
            return cam

        elif method == "full":
            cam, _ = self.add.relprop(cam, **kwargs)
            cam = cam[:, 1:]
            cam = self.patch_embed.relprop(cam, **kwargs)
            cam = cam.sum(dim=1)
            return cam
        elif method == "last_layer":
            cam = self.blocks[-1].attn.get_attn_cam()
            cam = cam[0].reshape(-1, cam.shape[-1], cam.shape[-1])
            if is_ablation:
                grad = self.blocks[-1].attn.get_attn_gradients()
                grad = grad[0].reshape(-1, grad.shape[-1], grad.shape[-1])
                cam = grad * cam
            cam = cam.clamp(min=0).mean(dim=0)
            cam = cam[0, 1:]
            return cam

        elif method == "last_layer_attn":
            cam = self.blocks[-1].attn.get_attn()
            cam = cam[0].reshape(-1, cam.shape[-1], cam.shape[-1])
            cam = cam.clamp(min=0).mean(dim=0)
            cam = cam[0, 1:]
            return cam

        elif method == "second_layer":
            cam = self.blocks[1].attn.get_attn_cam()
            cam = cam[0].reshape(-1, cam.shape[-1], cam.shape[-1])
            if is_ablation:
                grad = self.blocks[1].attn.get_attn_gradients()
                grad = grad[0].reshape(-1, grad.shape[-1], grad.shape[-1])
                cam = grad * cam
            cam = cam.clamp(min=0).mean(dim=0)
            cam = cam[0, 1:]
            return cam

    def relprop_from_features(self, cam, method="transformer_attribution", **kwargs):
        cam = self.norm.relprop(cam, **kwargs)
        for blk in reversed(self.blocks):
            cam = blk.relprop(cam, **kwargs)
        # (cam, _) = self.add.relprop(cam, **kwargs)
        cam = cam[:, 1:]
        cam = self.patch_embed.relprop(cam, **kwargs)
        return cam.sum(dim=1) if cam.dim() > 1 else cam

class CrossAttention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads, self.scale = num_heads, (dim // num_heads)**-0.5
        self.q_proj, self.kv_proj = Linear(dim, dim, bias=qkv_bias), Linear(dim, dim*2, bias=qkv_bias)
        self.attn_drop, self.proj, self.proj_drop = Dropout(attn_drop), Linear(dim, dim), Dropout(proj_drop)
        self.softmax = Softmax(dim=-1)
        self.matmul1 = einsum('bhid,bhjd->bhij')
        self.matmul2 = einsum('bhij,bhjd->bhid')
        self.attn_cam = self.attn_gradients = None
    def save_attn_cam(self, cam): self.attn_cam = cam
    def get_attn_cam(self): return self.attn_cam
    def save_attn_gradients(self, attn_gradients): self.attn_gradients = attn_gradients
    def get_attn_gradients(self): return self.attn_gradients
    def forward(self, x, context):
        B, N_q, C = x.shape; N_kv = context.shape[1]
        q = self.q_proj(x).reshape(B, N_q, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        kv = self.kv_proj(context).reshape(B, N_kv, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        dots = self.matmul1([q,k]) * self.scale
        attn = self.softmax(dots)
        attn = self.attn_drop(attn)
        attn.register_hook(self.save_attn_gradients)
        x = self.matmul2([attn,v]).transpose(1,2).reshape(B, N_q, C)
        return self.proj_drop(self.proj(x))
    
    def relprop(self, cam, **kwargs):
        cam = self.proj.relprop(self.proj_drop.relprop(cam, **kwargs), **kwargs)
        cam = rearrange(cam, 'b n (h d) -> b h n d', h=self.num_heads)
        (cam_attn, cam_v) = self.matmul2.relprop(cam, **kwargs)
        cam_attn /= 2; cam_v /= 2
        self.save_attn_cam(cam_attn)
        cam_attn = self.softmax.relprop(self.attn_drop.relprop(cam_attn, **kwargs), **kwargs)

        (cam_q, cam_k) = self.matmul1.relprop(cam_attn, **kwargs)
        cam_q /= 2; cam_k /= 2;
        cam_q = rearrange(cam_q, 'b h n d -> b n (h d)', h=self.num_heads)
        cam_x = self.q_proj.relprop(cam_q, **kwargs)
        cam_kv = rearrange([cam_k, cam_v], 'kv b h n d -> b n (kv h d)', kv=2, h=self.num_heads)
        cam_context = self.kv_proj.relprop(cam_kv, **kwargs)
        return cam_x, cam_context

class GPT2DecoderBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, drop=0., attn_drop=0.):
        super().__init__()
        self.norm1, self.attn = LayerNorm(dim, eps=1e-6), Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        self.norm2, self.cross_attn = LayerNorm(dim, eps=1e-6), CrossAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        self.norm3, self.mlp = LayerNorm(dim, eps=1e-6), Mlp(in_features=dim, hidden_features=int(dim * mlp_ratio), drop=drop)
        self.add1, self.add2, self.add3 = Add(), Add(), Add()
        self.clone1, self.clone2, self.clone3 = Clone(), Clone(), Clone()

    def forward(self, x, context):
        x1, x2 = self.clone1(x, 2); x = self.add1([x1, self.attn(self.norm1(x2))])
        x1, x2 = self.clone2(x, 2); x = self.add2([x1, self.cross_attn(self.norm2(x2), context)])
        x1, x2 = self.clone3(x, 2); x = self.add3([x1, self.mlp(self.norm3(x2))])
        return x

    def relprop(self, cam, context_cam, **kwargs):
        (cam1, cam2) = self.add3.relprop(cam, **kwargs);cam2 = self.mlp.relprop(self.norm3.relprop(cam2, **kwargs), **kwargs); cam = self.clone3.relprop((cam1, cam2), **kwargs)
        (cam1, cam2) = self.add2.relprop(cam, **kwargs); cam2, new_context_cam = self.cross_attn.relprop(cam2, **kwargs); cam2 = self.norm2.relprop(cam2, **kwargs); cam = self.clone2.relprop((cam1, cam2), **kwargs)
        context_cam += new_context_cam
        (cam1, cam2) = self.add1.relprop(cam, **kwargs); cam2 = self.attn.relprop(self.norm1.relprop(cam2, **kwargs), **kwargs); cam = self.clone1.relprop((cam1, cam2), **kwargs)
        return cam, context_cam
    

    

class VitGPT2Model(nn.Module):
    def __init__(self, encoder, vocab_size=10000, max_seq_len=77, embed_dim=768, depth=12, num_heads=12):
        super().__init__()
        self.encoder, self.embed_dim = encoder, embed_dim
        self.decoder_embed, self.decoder_pos_embed = TokenEncoder(vocab_size, embed_dim), nn.Parameter(torch.zeros(1, max_seq_len, embed_dim))
        self.decoder_blocks = nn.ModuleList([GPT2DecoderBlock(dim=embed_dim, num_heads=num_heads) for _ in range(depth)])
        self.decoder_norm, self.lm_head = LayerNorm(embed_dim), Linear(embed_dim, vocab_size, bias=False)
        self.add = Add()
        trunc_normal_(self.decoder_pos_embed, std=.02)
        
    def forward(self, image, caption):
        image_features = self.encoder.forward_features(image)
        token_embeddings = self.decoder_embed(caption)
        pos_embeddings = self.decoder_pos_embed[:, :caption.shape[1], :]
        x = self.add([token_embeddings, pos_embeddings])
        for blk in self.decoder_blocks: x = blk(x, image_features)
        x = self.decoder_norm(x)
        return self.lm_head(x)
    def relprop(self, cam, method='transformer_attribution', start_layer=0,initial_logit=None, **kwargs):
        if method == "transformer_attribution":
            _cam = self.lm_head.relprop(cam.clone(), **kwargs)
            _cam = self.decoder_norm.relprop(_cam, **kwargs)
            num_patches = self.encoder.patch_embed.num_patches
            _context_cam = torch.zeros(cam.shape[0], num_patches+1, self.embed_dim).to(cam.device)
            for blk in reversed(self.decoder_blocks):
                _cam, _context_cam = blk.relprop(_cam, _context_cam, **kwargs)
            (_cam, _) = self.add.relprop(_cam, **kwargs)
            _ = self.decoder_embed.relprop(_cam, **kwargs)
            self.encoder.relprop_from_features(_context_cam, **kwargs)
            encoder_attributions = [(blk.attn.get_attn_gradients()[0] * blk.attn.get_attn_cam()[0]).clamp(min=0).mean(dim=0).unsqueeze(0) for blk in self.encoder.blocks]
            encoder_rollout = compute_rollout_attention(encoder_attributions, start_layer=start_layer)
            target_token_idx = cam.shape[1] - 1
            cross_attribution_sum = torch.zeros(1,1,num_patches + 1).to(cam.device)
            for blk in self.decoder_blocks:
                grad = blk.cross_attn.get_attn_gradients()[0].reshape(blk.cross_attn.num_heads, -1, num_patches + 1)[:, target_token_idx, :].unsqueeze(1)
                cam_ = blk.cross_attn.get_attn_cam()[0].reshape(blk.cross_attn.num_heads, -1, num_patches + 1)[:, target_token_idx, :].unsqueeze(1)
                attr = (grad * cam_).clamp(min=0).mean(dim=0)
                cross_attribution_sum += attr
            final_cam = (cross_attribution_sum @ encoder_rollout).squeeze(0)
            return final_cam[:, 1:]
        elif method == 'full':
            if initial_logit is not None:
                print(f'\n --- LRP Conservation Check ---')
                print(f"Initial Logit Value: {initial_logit.item():.4f}")
            
            cam = self.lm_head.relprop(cam,**kwargs)
            print(f"After LM Head           {cam.sum().item():.4f}")
            
            cam = self.decoder_norm.relprop(cam, **kwargs)
            print(f"After Decoder Norm: {cam.sum().item():.4f}")
            num_patches = self.encoder.patch_embed.num_patches
            context_cam = torch.zeros(cam.shape[0], num_patches + 1, self.embed_dim).to(cam.device)
            for i, blk in enumerate(reversed(self.decoder_blocks)): 
                cam, context_cam = blk.relprop(cam, context_cam, **kwargs)
                print(f"After Decoder Block {len(self.decoder_blocks) - i - 1}: {(cam.sum() + context_cam.sum()).item():.4f} (Text: {cam.sum().item():.4f}, Image: {context_cam.sum().item():.4f})")
            (cam, positional_cam) = self.add.relprop(cam, **kwargs)
            print(f"After Embed+Pos Add:  Embedding: {(cam.sum()).item():.4f}, Position: {(positional_cam.sum()).item():.4f}")
            _ = self.decoder_embed.relprop(cam, **kwargs)
            print(f"Relevance to Encoder: {context_cam.sum().item():.4f}")
            final_map = self.encoder.relprop_from_features(context_cam, **kwargs)
            print(f"Final Map Sum:      {final_map.sum().item():.4f}")
            print(f"----------------------------------------")
            
            return final_map

# def _conv_filter(state_dict, patch_size=16):
#     out_dict = {}
#     for k,v in state_dict.items():
#         if 'patch_embed.proj.weight' in k:
#             v = v.reshape((v.shape[0], 3, patch_size, patch_size))
#         out_dict[k] = v
#     return out_dict

# def vit_base_patch16_224(pretrained=True, **kwargs):
#     model = VitTransformer(
#         patch_size=16, embed_dim=768, depth=12, num_heads=12, mlp_ratio=4, qkv_bias=True, **kwargs
#     )
#     model.default_cfg = default_cfgs['vit_base_patch16_224']
#     if pretrained:
#         load_pretrained(
#             model, num_classes=model.num_classes, in_chans=kwargs.get('in_chans', 3), filter_fn=_conv_filter
#         )
#         return model


class LRP:
    def __init__(self, model):
        self.model = model
        self.model.eval()

    def generate_LRP(self, input, caption, target_index=None, method='full', start_layer=0):
        output = self.model(input, caption)
        last_token_logits = output[:, -1, :]
        target_index = last_token_logits.argmax(dim=-1).item()
        one_hot = torch.zeros_like(last_token_logits)
        one_hot[:, target_index] = 1
        full_one_hot = torch.zeros_like(output)
        full_one_hot[:, -1, :] = one_hot
        self.model.zero_grad()
        output.backward(gradient=full_one_hot, retain_graph=True)
        return self.model.relprop(full_one_hot, method=method, start_layer=start_layer, alpha=1)
    

class Baselines:
    def __init__(self, model):
        self.model = model
        self.model.eval()

    def generate_cam_attn(self, input, index=None):
        output = self.model(input.cuda(), register_hook=True)
        if index == None:
            index = np.argmax(output.cpu().data.numpy())

        one_hot = np.zeros((1, output.size()[-1]), dtype=np.float32)
        one_hot[0][index] = 1
        one_hot = torch.from_numpy(one_hot).requires_grad_(True)
        one_hot = torch.sum(one_hot.cuda() * output)

        self.model.zero_grad()
        one_hot.backward(retain_graph=True)
        #################### attn
        grad = self.model.blocks[-1].attn.get_attn_gradients()
        cam = self.model.blocks[-1].attn.get_attention_map()
        cam = cam[0, :, 0, 1:].reshape(-1, 14, 14)
        grad = grad[0, :, 0, 1:].reshape(-1, 14, 14)
        grad = grad.mean(dim=[1, 2], keepdim=True)
        cam = (cam * grad).mean(0).clamp(min=0)
        cam = (cam - cam.min()) / (cam.max() - cam.min())

        return cam
        #################### attn

    def generate_rollout(self, input, start_layer=0):
        self.model(input)
        blocks = self.model.blocks
        all_layer_attentions = []
        for blk in blocks:
            attn_heads = blk.attn.get_attention_map()
            avg_heads = (attn_heads.sum(dim=1) / attn_heads.shape[1]).detach()
            all_layer_attentions.append(avg_heads)
        rollout = compute_rollout_attention(all_layer_attentions, start_layer=start_layer)
        return rollout[:,0, 1:]
    

vit_encoder = VitTransformer()
model = VitGPT2Model(encoder=vit_encoder, vocab_size=10000, depth=4)
model.eval()
dummy_image = torch.randn(1, 3, 224, 224)
dummy_caption = torch.randint(0, 10000, (1, 10))
lrp_generator = LRP(model)
attribution_full = lrp_generator.generate_LRP(dummy_image, dummy_caption, method='full')
print(f"Shape of 'full' attribution map: {attribution_full.shape}")
print("\n--- Generating with 'transformer_attribution' method ---")
attribution_transformer = lrp_generator.generate_LRP(dummy_image, dummy_caption, method='transformer_attribution')
print(f"Shape of 'transformer_attribution' map: {attribution_transformer.shape}")
print(model)
    
    




